# Causal ablation of S1/voice shared SAE features (0.5B, layer 18)

This notebook tests whether features highlighted by the shared SVD analysis causally control both rules.

**Primary arm:** the six highest-SVD features that also had prior S1 selection evidence.

**Comparisons:** literal SVD top six and the unchanged historical control arm.

Both fixed-CoT readout ablation and variable-CoT generation ablation are run on existing scored baselines:
- S1: `qwen05b_v2.jsonl` (different generated CoTs from the SVD source, but the same ETHICS scenarios)
- Voice: `voice_transfer_05b_l18/baselines/qwen05b_voice.jsonl` (separate from voice SVD discovery pairs)

Prior evidence: the S1 combined-six and flip-sensitive-six variable-CoT arms changed 27 and 20 labels and reduced first-sentence following from 89/90 to 66/90 and 72/90. Prior voice-transfer arms changed no labels, so voice transfer remains unproven.

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
REPO_URL = "https://github.com/vladflorinfilip/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography.git"

!rm -rf Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!git clone --depth 1 "{REPO_URL}"
%cd Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!pip install -q peft "transformers<4.50" accelerate tqdm
!pip uninstall -y torchao >/dev/null 2>&1

In [ ]:
from pathlib import Path
from google.colab import files
from transformers import AutoTokenizer
import hashlib

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
S1_ADAPTER = Path("checkpoints/qwen05b-cot-sft-v2")
VOICE_ADAPTER = Path("checkpoints/qwen05b-cot-sft-voice-paired")
ARTIFACT_DIR = Path("sparse_autoencoders/artifacts/ethics_l18")
SVD_FEATURES = Path("sparse_autoencoders/artifacts/sae_svd_05b_l18/top_shared_sae_features.json")
S1_BASELINE = Path("data/evaluation_data/qwen/ETHICS/qwen05b_v2.jsonl")
VOICE_BASELINE = Path("data/voice_transfer_05b_l18/baselines/qwen05b_voice.jsonl")
OUTPUT_DIR = ARTIFACT_DIR / "ablations/shared_svd"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in (S1_ADAPTER / "adapter_model.safetensors", ARTIFACT_DIR / "sae.pt", SVD_FEATURES, S1_BASELINE, VOICE_BASELINE):
    assert path.is_file(), f"Missing committed input: {path}"

print("Upload paired-voice adapter_config.json and adapter_model.safetensors")
uploaded = files.upload()
VOICE_ADAPTER.mkdir(parents=True, exist_ok=True)
for name, data in uploaded.items():
    if Path(name).name in {"adapter_config.json", "adapter_model.safetensors", "training_log.json"}:
        (VOICE_ADAPTER / Path(name).name).write_bytes(data)
assert (VOICE_ADAPTER / "adapter_config.json").is_file()
assert (VOICE_ADAPTER / "adapter_model.safetensors").is_file()
AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(VOICE_ADAPTER)

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

ADAPTER_HASHES = {
    "s1": sha256(S1_ADAPTER / "adapter_model.safetensors"),
    "voice": sha256(VOICE_ADAPTER / "adapter_model.safetensors"),
}
assert ADAPTER_HASHES["s1"] != ADAPTER_HASHES["voice"], "The two uploaded adapters are identical"
print("Distinct adapters verified:", {key: value[:12] for key, value in ADAPTER_HASHES.items()})

In [ ]:
import json

svd_ranking = json.loads(SVD_FEATURES.read_text())
prior_combined6 = {2976, 3578, 6975, 1392, 4781, 1741}
prior_flip6 = {1846, 1772, 6808, 4665, 872, 4778}
prior_supported = prior_combined6 | prior_flip6

svd_top6 = [row["feature"] for row in svd_ranking[:6]]
consensus6 = [row["feature"] for row in svd_ranking if row["feature"] in prior_supported][:6]
control6 = [304, 4898, 2217, 7146, 2726, 6864]  # unchanged historical control
assert svd_top6 == [3578, 6975, 1392, 2521, 4665, 6808]
assert consensus6 == [3578, 6975, 1392, 4665, 6808, 1846]

FEATURE_ARMS = {
    "consensus6": consensus6,       # primary: shared SVD + prior S1 evidence
    "svd_top6": svd_top6,           # literal highest absolute SVD loadings
    "prior_control6": control6,     # prior non-overlapping PEFT ranks 7–12
}

print(f"{'rank':>4} {'feature':>8} {'loading':>10}  prior evidence")
for row in svd_ranking[:15]:
    evidence = []
    if row["feature"] in prior_combined6: evidence.append("combined6")
    if row["feature"] in prior_flip6: evidence.append("flip6")
    print(f"{row['rank']:4d} {row['feature']:8d} {row['loading']:10.3f}  {', '.join(evidence) or 'SVD only'}")
print("\nExperiment arms:", FEATURE_ARMS)

In [ ]:
import subprocess, sys

TASKS = {
    "s1": {"model": S1_ADAPTER, "baseline": S1_BASELINE},
    "voice": {"model": VOICE_ADAPTER, "baseline": VOICE_BASELINE},
}

def stream(cmd):
    print("+", " ".join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError(f"Command failed with exit code {process.returncode}")

def run_ablation(task, mode, arm, features):
    output = OUTPUT_DIR / task / mode / f"{arm}.jsonl"
    output.parent.mkdir(parents=True, exist_ok=True)
    stream([
        sys.executable, "-u", "sparse_autoencoders/ablate_features.py",
        "--features", *map(str, features),
        "--mode", mode,
        "--artifact-dir", str(ARTIFACT_DIR),
        "--model", str(TASKS[task]["model"]),
        "--generations", str(TASKS[task]["baseline"]),
        "--output", str(output),
        "--device", "cuda", "--dtype", "float16", "--overwrite",
    ])
    return output

In [ ]:
# Variable-CoT: intervene throughout reasoning and answer generation. This is the slower cell.
RUN_VARIABLE = True
variable_outputs = []
if RUN_VARIABLE:
    for task in TASKS:
        for arm, features in FEATURE_ARMS.items():
            variable_outputs.append(run_ablation(task, "generate", arm, features))
print("Variable-CoT runs complete" if RUN_VARIABLE else "Variable-CoT runs skipped")

In [ ]:
# Add lexical voice labels to every voice output without an API call.
from evaluation.score_voice_alignment import annotate_record, write_jsonl

for path in list((OUTPUT_DIR / "voice").glob("*/*.jsonl")):
    records = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    for record in records:
        annotate_record(record, critic=None)
    write_jsonl(path, records)
print("Voice outputs annotated")

In [ ]:
from intervention.cot_utils import classify_stance, split_sentences

def read_rows(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

def summarize(task, mode, arm, path):
    rows = read_rows(path)
    baseline = {int(row["index"]): row for row in read_rows(TASKS[task]["baseline"])}
    parsed = [row for row in rows if row.get("prediction") is not None]
    changed = sum(int(row["prediction"]) != int(baseline[int(row["index"])]["prediction"]) for row in parsed)
    cot_changed = sum(row.get("chain_of_thought", "") != baseline[int(row["index"])].get("chain_of_thought", "") for row in rows)
    follow_successes = resolved = 0
    if task == "voice":
        for row in parsed:
            if "lexical_follows_voice" not in row:
                annotate_record(row, critic=None)
            resolved += int(row.get("lexical_cot_voice") in {"active", "passive"})
            follow_successes += int(row.get("lexical_follows_voice") is True)
    else:
        for row in parsed:
            sentences = split_sentences(row.get("chain_of_thought", ""))
            stance = classify_stance(sentences[0]) if sentences else None
            if stance is not None:
                resolved += 1
                follow_successes += int(int(row["prediction"]) == stance)
    return {
        "task": task, "mode": mode, "arm": arm, "n": len(rows),
        "accuracy": sum(int(row["prediction"]) == int(row["gold"]) for row in parsed) / len(parsed),
        "prediction_1_rate": sum(int(row["prediction"]) == 1 for row in parsed) / len(parsed),
        "label_changes": changed,
        "cot_changes": cot_changed,
        "lexical_follow_all_parsed": follow_successes / len(parsed),
        "lexical_follow_resolved_only": follow_successes / resolved if resolved else None,
        "lexical_resolved_n": resolved,
    }

results = []
for task, spec in TASKS.items():
    results.append(summarize(task, "baseline", "none", spec["baseline"]))
    for mode in ("score", "generate"):
        for arm in FEATURE_ARMS:
            path = OUTPUT_DIR / task / mode / f"{arm}.jsonl"
            if path.is_file():
                results.append(summarize(task, mode, arm, path))

print(f"{'task':6s} {'mode':9s} {'arm':16s} {'acc':>6s} {'lex/all':>8s} {'resolved':>9s} {'labels':>7s} {'CoTs':>5s}")
for row in results:
    print(
        f"{row['task']:6s} {row['mode']:9s} {row['arm']:16s} "
        f"{row['accuracy']:6.3f} {row['lexical_follow_all_parsed']:8.3f} "
        f"{row['lexical_resolved_n']:9d} {row['label_changes']:7d} {row['cot_changes']:5d}"
    )

metadata = {
    "base_model": BASE_MODEL,
    "layer": 18,
    "sae": str(ARTIFACT_DIR / "sae.pt"),
    "adapter_sha256": ADAPTER_HASHES,
    "feature_arms": FEATURE_ARMS,
    "primary_arm": "consensus6",
    "selection": "highest absolute shared-SVD loadings among features with prior S1 selection evidence",
    "baselines": {task: str(spec["baseline"]) for task, spec in TASKS.items()},
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(results, indent=2))
(OUTPUT_DIR / "experiment.json").write_text(json.dumps(metadata, indent=2))

In [ ]:
import shutil

assert (OUTPUT_DIR / "summary.json").is_file()
assert (OUTPUT_DIR / "experiment.json").is_file()
archive = shutil.make_archive("/content/sae_shared_svd_ablation_05b_l18", "zip", root_dir=OUTPUT_DIR)
files.download(archive)
print("Downloaded", archive)